In [0]:
WITH bronze_summary AS (
  SELECT
    COUNT(*) AS bronze_count
  FROM nyc_taxi.bronze.bronze_taxi_zone_lookup
),

silver_summary AS (
  SELECT
    COUNT(*) AS silver_count,
    COUNT(DISTINCT location_id)
      AS distinct_location_ids,
    COUNT_IF(location_id IS NULL)
      AS null_location_ids,
    COUNT_IF(
      borough IS NULL
      OR TRIM(borough) = ''
    ) AS missing_borough,
    COUNT_IF(
      zone IS NULL
      OR TRIM(zone) = ''
    ) AS missing_zone,
    COUNT_IF(
      service_zone IS NULL
      OR TRIM(service_zone) = ''
    ) AS missing_service_zone
  FROM nyc_taxi.silver.silver_taxi_zone_lookup
),

duplicates AS (
  SELECT
    COUNT(*) AS duplicated_location_ids
  FROM (
    SELECT location_id
    FROM nyc_taxi.silver.silver_taxi_zone_lookup
    GROUP BY location_id
    HAVING COUNT(*) > 1
  )
)

SELECT
  b.bronze_count,
  s.silver_count,
  s.silver_count - b.bronze_count
    AS count_difference,
  s.distinct_location_ids,
  s.null_location_ids,
  d.duplicated_location_ids,
  s.missing_borough,
  s.missing_zone,
  s.missing_service_zone,

  CASE
    WHEN b.bronze_count <> s.silver_count
      THEN 'COUNT_MISMATCH'
    WHEN s.null_location_ids > 0
      THEN 'NULL_LOCATION_ID'
    WHEN d.duplicated_location_ids > 0
      THEN 'DUPLICATED_LOCATION_ID'
    WHEN s.missing_borough > 0
      OR s.missing_zone > 0
      OR s.missing_service_zone > 0
      THEN 'MISSING_ATTRIBUTE'
    ELSE 'SILVER_ZONE_LOOKUP_VALIDATED'
  END AS validation_status

FROM bronze_summary AS b
CROSS JOIN silver_summary AS s
CROSS JOIN duplicates AS d;

WITH bronze_summary AS (
  SELECT
    COUNT(*) AS bronze_count
  FROM nyc_taxi.bronze.bronze_yellow_trip_2025
),

silver_summary AS (
  SELECT
    COUNT(*) AS silver_count,

    COUNT_IF(
      pickup_datetime IS NULL
      OR dropoff_datetime IS NULL
      OR dropoff_datetime <= pickup_datetime
    ) AS invalid_trip_period,

    COUNT_IF(
      YEAR(pickup_datetime) <> 2025
      OR source_year <> 2025
      OR source_month <> MONTH(pickup_datetime)
    ) AS invalid_source_period,

    COUNT_IF(
      trip_distance IS NULL
      OR trip_distance < 0
    ) AS invalid_trip_distance,

    COUNT_IF(
      total_amount IS NULL
    ) AS missing_total_amount,

    COUNT_IF(
      pickup_location_id IS NULL
      OR pickup_zone_location_id IS NULL
    ) AS unresolved_pickup_locations,

    COUNT_IF(
      dropoff_location_id IS NULL
      OR dropoff_zone_location_id IS NULL
    ) AS unresolved_dropoff_locations,

    COUNT_IF(
      _is_quarantined
      OR SIZE(_dq_reasons) > 0
    ) AS incorrectly_published_records,

    COUNT_IF(_is_zero_distance)
      AS zero_distance_records,

    COUNT_IF(_is_passenger_count_missing)
      AS missing_passenger_count_records,

    COUNT_IF(_is_non_positive_passenger_count)
      AS non_positive_passenger_records,

    COUNT_IF(_is_negative_total_amount)
      AS negative_total_amount_records,

    COUNT_IF(_is_long_trip)
      AS long_trip_records,

    COUNT(*) - COUNT(DISTINCT _record_hash)
      AS potential_duplicate_records

  FROM nyc_taxi.silver.silver_yellow_trip_2025
),

quarantine_summary AS (
  SELECT
    COUNT(*) AS quarantine_count,

    COUNT_IF(
      NOT _is_quarantined
      OR COALESCE(SIZE(_dq_reasons), 0) = 0
    ) AS invalid_quarantine_classification

  FROM nyc_taxi.silver.quarantine_yellow_trip_2025
)

SELECT
  b.bronze_count,
  s.silver_count,
  q.quarantine_count,

  s.silver_count + q.quarantine_count
    AS classified_count,

  b.bronze_count
    - (s.silver_count + q.quarantine_count)
    AS reconciliation_difference,

  s.invalid_trip_period,
  s.invalid_source_period,
  s.invalid_trip_distance,
  s.missing_total_amount,
  s.unresolved_pickup_locations,
  s.unresolved_dropoff_locations,
  s.incorrectly_published_records,
  q.invalid_quarantine_classification,

  s.zero_distance_records,
  s.missing_passenger_count_records,
  s.non_positive_passenger_records,
  s.negative_total_amount_records,
  s.long_trip_records,
  s.potential_duplicate_records,

  CASE
    WHEN b.bronze_count
       <> s.silver_count + q.quarantine_count
      THEN 'RECONCILIATION_FAILED'

    WHEN s.invalid_trip_period > 0
      OR s.invalid_source_period > 0
      OR s.invalid_trip_distance > 0
      OR s.missing_total_amount > 0
      THEN 'INVALID_RECORD_IN_SILVER'

    WHEN s.unresolved_pickup_locations > 0
      OR s.unresolved_dropoff_locations > 0
      THEN 'UNRESOLVED_LOCATION_IN_SILVER'

    WHEN s.incorrectly_published_records > 0
      THEN 'QUARANTINED_RECORD_IN_SILVER'

    WHEN q.invalid_quarantine_classification > 0
      THEN 'INVALID_QUARANTINE_CLASSIFICATION'

    ELSE 'SILVER_TAXI_TRIP_VALIDATED'
  END AS validation_status

FROM bronze_summary AS b
CROSS JOIN silver_summary AS s
CROSS JOIN quarantine_summary AS q;